# Searching

Notes and runnable examples on searching — finding an element, or where it belongs. The idea that ties it together: **how fast you can search is decided by what structure you've already imposed on the data.** Nothing → O(n) scan; sorted → O(log n) binary search; hashed → O(1). We'll build binary search carefully (it's famously easy to get wrong), then see how `bisect` and the "binary search on the answer" trick generalize it.

**Contents**

1. **What searching is** — structure decides speed
2. **Linear search**
3. **Binary search from scratch**
4. **`bisect`** — Python's C binary search
5. **Binary search on the answer** — searching a monotonic predicate
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. What searching is — structure decides speed

"Search" just means: *is x here, and where?* How fast you can answer depends entirely on the **structure you've already paid for**:

| What you've got | Search cost | How |
|---|:---:|---|
| an unordered sequence | **O(n)** | scan every element (linear search) |
| a **sorted** sequence | **O(log n)** | halve the range each step (binary search) |
| a **hash** table (`set` / `dict`) | **O(1)** avg | jump straight to the slot |
| a balanced **tree** | **O(log n)** | follow comparisons down |

So the interesting choice is usually made *earlier* — when you decide how to store the data. Sorting (the previous notebook) and hashing (the dict & set notebook) are really **investments that buy cheap search later**.

## 2. Linear search

The baseline: walk the sequence and return the first match. It needs **no precondition** — works on any iterable, sorted or not — which is exactly why it's O(n). For a *one-off* lookup in unsorted data it's also the right answer: sorting first (O(n log n)) only pays off if you'll search many times.

In [1]:
def linear_search(xs, target):
    for i, x in enumerate(xs):
        if x == target:
            return i
    return -1

data = [4, 1, 9, 2, 7]
print("find 9:", linear_search(data, 9))     # 2
print("find 5:", linear_search(data, 5))     # -1

# Python's `in` and .index() are linear search in C:
print("9 in data:", 9 in data, "| data.index(9):", data.index(9))


find 9: 2
find 5: -1
9 in data: True | data.index(9): 2


## 3. Binary search from scratch

If the data is **sorted**, you can do far better: check the middle element, and since you know which half the target must lie in, throw the other half away. Each step halves the search space, so it finishes in **O(log n)** — about 20 steps for a million elements.

It's also *notoriously* easy to get wrong — the bugs live in the boundary conditions. The reliable formulation keeps an **inclusive** range `[lo, hi]` and the invariant "*if the target is present, it's somewhere in `[lo, hi]`*":

In [2]:
def binary_search(a, target):
    lo, hi = 0, len(a) - 1
    while lo <= hi:                 # inclusive range; stop when it becomes empty
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid
        elif a[mid] < target:
            lo = mid + 1            # target must be in the right half
        else:
            hi = mid - 1            # target must be in the left half
    return -1

a = list(range(1_000_000))
print("find 999_999:", binary_search(a, 999_999))
print("find -1     :", binary_search(a, -1))


find 999_999: 999999
find -1     : -1


A million-element search touches only ~20 elements (log₂ 10⁶ ≈ 20). One Python-specific nicety: the classic binary-search bug in C/Java — `mid = (lo + hi) / 2` **overflowing** when `lo + hi` exceeds the integer max — *can't happen in Python*, whose `int` is arbitrary-precision (see the planned bit-manipulation notebook). So a plain `(lo + hi) // 2` is always safe here.

## 4. `bisect` — Python's C binary search

You rarely write that loop; the stdlib **`bisect`** module is binary search in C. Rather than "found / not found," it returns the **insertion point** — where a value belongs to keep the list sorted — which turns out to be more useful: it powers membership, counting, predecessor/successor, and range queries.

The two variants differ only on **ties**:

- `bisect_left(a, x)` → leftmost spot (before any equal elements)
- `bisect_right(a, x)` → rightmost spot (after them)

In [3]:
import bisect

a = [1, 2, 2, 2, 3, 5]
print("bisect_left (a, 2):", bisect.bisect_left(a, 2))    # 1
print("bisect_right(a, 2):", bisect.bisect_right(a, 2))   # 4
print("count of 2        :", bisect.bisect_right(a, 2) - bisect.bisect_left(a, 2))   # 3

# membership test in O(log n):
def contains(a, x):
    i = bisect.bisect_left(a, x)
    return i < len(a) and a[i] == x
print("contains 3:", contains(a, 3), "| contains 4:", contains(a, 4))

# predecessor / successor around a value that isn't present:
b = [10, 20, 30, 40, 50]
i = bisect.bisect_left(b, 35)
print("around 35 -> pred:", b[i - 1], "succ:", b[i])

# since Python 3.10, bisect takes a key= (the list must be sorted by that key):
rows = [("a", 1), ("b", 3), ("c", 5)]
print("by 2nd field, find 3:", bisect.bisect_left(rows, 3, key=lambda r: r[1]))   # 1

# bisect.insort places a value in O(log n) search + O(n) shift:
bisect.insort(a, 4)
print("after insort(4)   :", a)


bisect_left (a, 2): 1
bisect_right(a, 2): 4
count of 2        : 3
contains 3: True | contains 4: False
around 35 -> pred: 30 succ: 40
by 2nd field, find 3: 1
after insort(4)   : [1, 2, 2, 2, 3, 4, 5]


## 5. Binary search on the answer

Binary search isn't really about arrays — it's about any **monotonic** yes/no question. If a predicate goes False, False, …, False, **True**, True, …, True over a range, you can binary-search for that boundary without materializing anything. This *"binary search on the answer"* is one of the most useful problem-solving patterns — it turns "search a million possibilities" into ~20 predicate checks.

The reusable core finds the first `x` where `pred(x)` is true:

In [4]:
def first_true(lo, hi, pred):
    """Smallest x in [lo, hi] with pred(x) True (pred must be monotonic: F...F T...T)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid                # the answer is mid, or somewhere to its left
        else:
            lo = mid + 1            # the answer is strictly to the right
    return lo

# integer square root: largest k with k*k <= n  ==  (first k with k*k > n) - 1
import math
n = 1_000_000
isqrt = first_true(0, n, lambda k: k * k > n) - 1
print("integer sqrt of 1e6:", isqrt, "| math.isqrt:", math.isqrt(n))

# the same shape solves "first bad version", capacity/threshold, min-feasible-X problems:
versions = [False, False, False, True, True, True]       # monotonic
print("first bad version  :", first_true(0, len(versions) - 1, lambda i: versions[i]))


integer sqrt of 1e6: 1000 | math.isqrt: 1000
first bad version  : 3


## 6. When to use what

| Situation | Reach for | Cost |
|---|---|---|
| Unsorted data, one-off lookup | linear scan / `in` | O(n) |
| Many membership tests | `set` / `dict` | O(1) avg — convert once |
| Sorted list: find / count / range | `bisect` | O(log n) |
| Boundary of a monotonic condition | binary search on the answer | O(log R · P) |
| Lookup by key, ordered | `dict` (O(1)) or `sortedcontainers` (O(log n)) | — |

**Don't binary-search data you touch once:** the O(n log n) sort to enable it costs more than a single O(n) scan. Binary search pays off when you search *repeatedly*, or when the data arrives already sorted.

## 7. Complexity cheat-sheet

| Method | Precondition | Time | Notes |
|---|---|:---:|---|
| Linear search (`in`, `.index`) | none | O(n) | works on anything |
| Binary search / `bisect` | sorted | O(log n) | + O(n) to keep sorted on insert |
| Hash lookup (`set` / `dict`) | hashable keys | O(1) avg, O(n) worst | no ordering |
| BST / tree search | sorted tree | O(log n) balanced, O(n) skewed | see trees notebook |
| Binary search on the answer | monotonic predicate | O(log R · P) | R = range size, P = predicate cost |